In [1]:
"""
================================================================================
TRAIN GCN WITH MINI-BATCH SAMPLING - FINAL FIXED VERSION
================================================================================
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import scipy.sparse as sp
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm import tqdm
import json
import time
import gc

print("="*80)
print("TRAINING GCN WITH MINI-BATCH SAMPLING")
print("="*80)

# ============================================================================
# GPU CLEANUP
# ============================================================================
print("\nCleaning GPU memory...")
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    gc.collect()
    print(f"✓ GPU memory cleared")
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory allocated: {torch.cuda.memory_allocated(0) / 1024**2:.1f} MB")
else:
    print("⚠ CUDA not available - using CPU")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nDevice: {device}")

# ============================================================================
# CONFIGURATION  
# ============================================================================
class Config:
    ADJ_PATH = 'heterogeneous_adj.npz'
    FEATURES_PATH = 'heterogeneous_features.npy'
    LABELS_PATH = 'heterogeneous_labels.npy'
    TRAIN_MASK_PATH = 'heterogeneous_train_mask.npy'
    VAL_MASK_PATH = 'heterogeneous_val_mask.npy'
    TEST_MASK_PATH = 'heterogeneous_test_mask.npy'
    NODE_MAPPING_PATH = 'node_mapping.json'
    
    HIDDEN_DIM = 256
    DROPOUT = 0.5
    LEARNING_RATE = 0.01
    WEIGHT_DECAY = 5e-4
    EPOCHS = 200
    
    BATCH_SIZE = 512
    NUM_NEIGHBORS = [10, 5]
    
    MODEL_SAVE_PATH = 'best_gcn_minibatch.pt'
    RESULTS_PATH = 'gcn_minibatch_results.json'

config = Config()

# ============================================================================
# LOAD DATA
# ============================================================================
print("\n" + "="*80)
print("LOADING DATA")
print("="*80)

adj = sp.load_npz(config.ADJ_PATH)
features = np.load(config.FEATURES_PATH)
labels = np.load(config.LABELS_PATH)
train_mask = np.load(config.TRAIN_MASK_PATH)
val_mask = np.load(config.VAL_MASK_PATH)
test_mask = np.load(config.TEST_MASK_PATH)

with open(config.NODE_MAPPING_PATH, 'r') as f:
    mapping = json.load(f)

n_docs = mapping['n_docs']
n_nodes = mapping['n_nodes']

print(f"✓ Nodes: {n_nodes:,}, Edges: {adj.nnz:,}")
print(f"✓ Features: {features.shape}")
print(f"✓ Train: {train_mask.sum():,}, Val: {val_mask.sum():,}, Test: {test_mask.sum():,}")

# ============================================================================
# LABEL HANDLING
# ============================================================================
doc_labels = labels[:n_docs]
unique_labels = np.unique(doc_labels)
num_classes = len(unique_labels)

print(f"✓ Number of classes: {num_classes}")
print(f"✓ Label range: [{unique_labels.min()}, {unique_labels.max()}]")

if unique_labels.min() != 0 or unique_labels.max() != num_classes - 1:
    print("⚠ Remapping labels...")
    label_map = {old: new for new, old in enumerate(sorted(unique_labels))}
    for i in range(n_docs):
        labels[i] = label_map[labels[i]]
    print(f"✓ Labels remapped")
else:
    print("✓ Labels already in [0, num_classes-1]")

assert labels[:n_docs].min() >= 0
assert labels[:n_docs].max() < num_classes
print("✓ Label validation passed")

# ============================================================================
# PREPARE TENSORS
# ============================================================================
print("\nPreparing tensors...")
features_tensor = torch.FloatTensor(features)
labels_tensor = torch.LongTensor(labels)

del features, labels
gc.collect()

adj_csr = adj.tocsr()
print("✓ Tensors prepared")

# ============================================================================
# GCN MODEL
# ============================================================================

class GCN(nn.Module):
    def __init__(self, n_features, n_hidden, n_classes, dropout=0.5):
        super().__init__()
        self.fc1 = nn.Linear(n_features, n_hidden)
        self.fc2 = nn.Linear(n_hidden, n_classes)
        self.dropout = dropout
    
    def forward(self, x, adj):
        x = self.fc1(x)
        x = torch.sparse.mm(adj, x)
        x = F.relu(x)
        x = F.dropout(x, self.dropout, training=self.training)
        
        x = self.fc2(x)
        x = torch.sparse.mm(adj, x)
        
        return F.log_softmax(x, dim=1)

print("\nInitializing model...")
model = GCN(
    n_features=features_tensor.shape[1],
    n_hidden=config.HIDDEN_DIM,
    n_classes=num_classes,
    dropout=config.DROPOUT
)

print(f"✓ Model created on CPU")
print(f"  Input: {features_tensor.shape[1]}, Hidden: {config.HIDDEN_DIM}, Output: {num_classes}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

if device.type == 'cuda':
    try:
        print("\nMoving model to GPU...")
        model = model.to(device)
        print("✓ Model on GPU")
        print(f"  GPU memory: {torch.cuda.memory_allocated(0) / 1024**2:.1f} MB")
    except RuntimeError as e:
        print(f"❌ Error: {e}")
        print("Falling back to CPU")
        device = torch.device('cpu')

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

print(f"✓ Optimizer initialized")

# ============================================================================
# SAMPLING FUNCTIONS
# ============================================================================

def sample_neighbors(adj_csr, nodes, num_neighbors):
    sampled = []
    for node in nodes:
        neighbors = adj_csr[node].indices
        if len(neighbors) > num_neighbors:
            neighbors = np.random.choice(neighbors, num_neighbors, replace=False)
        if len(neighbors) > 0:
            sampled.append(neighbors)
    return np.unique(np.concatenate(sampled)) if sampled else np.array([])

def create_subgraph(adj_csr, nodes):
    sub_adj = adj_csr[nodes][:, nodes]
    
    rowsum = np.array(sub_adj.sum(1)).flatten()
    d_inv_sqrt = np.power(rowsum, -0.5)
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.
    d_mat = sp.diags(d_inv_sqrt)
    sub_adj_norm = d_mat @ sub_adj @ d_mat
    
    sub_adj_coo = sub_adj_norm.tocoo()
    indices = torch.LongTensor(np.vstack((sub_adj_coo.row, sub_adj_coo.col)))
    values = torch.FloatTensor(sub_adj_coo.data)
    shape = torch.Size(sub_adj_coo.shape)
    
    return torch.sparse_coo_tensor(indices, values, shape)

# ============================================================================
# TRAINING
# ============================================================================

print("\n" + "="*80)
print("TRAINING")
print("="*80)

best_val_f1 = 0.0
best_val_acc = 0.0
train_nodes = np.where(train_mask[:n_docs])[0]

print(f"Training on {len(train_nodes):,} document nodes")
print(f"Batches per epoch: {(len(train_nodes) + config.BATCH_SIZE - 1) // config.BATCH_SIZE}")
print()

for epoch in range(config.EPOCHS):
    model.train()
    np.random.shuffle(train_nodes)
    
    epoch_start = time.time()
    total_loss = 0
    num_batches = 0
    
    num_train_batches = (len(train_nodes) + config.BATCH_SIZE - 1) // config.BATCH_SIZE
    pbar = tqdm(range(0, len(train_nodes), config.BATCH_SIZE), 
                desc=f'Epoch {epoch+1:3d}/{config.EPOCHS}',
                total=num_train_batches,
                leave=False)
    
    for i in pbar:
        batch_nodes = train_nodes[i:i + config.BATCH_SIZE]
        
        hop1 = sample_neighbors(adj_csr, batch_nodes, config.NUM_NEIGHBORS[0])
        hop2 = sample_neighbors(adj_csr, hop1, config.NUM_NEIGHBORS[1]) if len(hop1) > 0 else np.array([])
        all_nodes = np.unique(np.concatenate([batch_nodes, hop1, hop2]))
        
        sub_adj = create_subgraph(adj_csr, all_nodes).to(device)
        sub_features = features_tensor[all_nodes].to(device)
        
        node_map = {old: new for new, old in enumerate(all_nodes)}
        batch_indices = torch.LongTensor([node_map[n] for n in batch_nodes]).to(device)
        batch_labels = labels_tensor[batch_nodes].to(device)
        
        optimizer.zero_grad()
        output = model(sub_features, sub_adj)
        loss = F.nll_loss(output[batch_indices], batch_labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        del sub_adj, sub_features, output, batch_labels
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    
    avg_loss = total_loss / num_batches if num_batches > 0 else 0
    epoch_time = time.time() - epoch_start
    
    # Validation every 10 epochs
    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            all_preds = torch.zeros(n_nodes, dtype=torch.long)
            val_nodes = np.where(val_mask[:n_docs])[0]
            
            for i in tqdm(range(0, len(val_nodes), config.BATCH_SIZE), 
                         desc='Validating', leave=False):
                batch = val_nodes[i:i + config.BATCH_SIZE]
                hop1 = sample_neighbors(adj_csr, batch, config.NUM_NEIGHBORS[0])
                hop2 = sample_neighbors(adj_csr, hop1, config.NUM_NEIGHBORS[1]) if len(hop1) > 0 else np.array([])
                nodes = np.unique(np.concatenate([batch, hop1, hop2]))
                
                sub_adj = create_subgraph(adj_csr, nodes).to(device)
                sub_features = features_tensor[nodes].to(device)
                
                output = model(sub_features, sub_adj)
                node_map = {old: new for new, old in enumerate(nodes)}
                batch_idx = [node_map[n] for n in batch]
                all_preds[batch] = output[batch_idx].argmax(dim=1).cpu()
                
                del sub_adj, sub_features, output
            
            # FIX: Apply mask to full all_preds, not sliced version
            val_pred = all_preds[val_mask]
            val_true = labels_tensor[val_mask]
            val_acc = accuracy_score(val_true, val_pred)
            val_f1 = f1_score(val_true, val_pred, average='weighted')
            
            status = '✓ NEW BEST' if val_f1 > best_val_f1 else ''
            print(f'Epoch {epoch+1:3d} | Time: {epoch_time:.1f}s | Loss: {avg_loss:.4f} | '
                  f'Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f} {status}')
            
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                best_val_acc = val_acc
                torch.save(model.state_dict(), config.MODEL_SAVE_PATH)
    else:
        print(f'Epoch {epoch+1:3d} | Time: {epoch_time:.1f}s | Loss: {avg_loss:.4f}')

print("\n✅ Training complete!")
print(f"Best Val Acc: {best_val_acc:.4f}, Best Val F1: {best_val_f1:.4f}")

# ============================================================================
# TEST
# ============================================================================
print("\n" + "="*80)
print("TEST EVALUATION")
print("="*80)

model.load_state_dict(torch.load(config.MODEL_SAVE_PATH))
model.eval()

with torch.no_grad():
    all_preds = torch.zeros(n_nodes, dtype=torch.long)
    test_nodes = np.where(test_mask[:n_docs])[0]
    
    for i in tqdm(range(0, len(test_nodes), config.BATCH_SIZE), desc="Testing"):
        batch = test_nodes[i:i + config.BATCH_SIZE]
        hop1 = sample_neighbors(adj_csr, batch, config.NUM_NEIGHBORS[0])
        hop2 = sample_neighbors(adj_csr, hop1, config.NUM_NEIGHBORS[1]) if len(hop1) > 0 else np.array([])
        nodes = np.unique(np.concatenate([batch, hop1, hop2]))
        
        sub_adj = create_subgraph(adj_csr, nodes).to(device)
        sub_features = features_tensor[nodes].to(device)
        
        output = model(sub_features, sub_adj)
        node_map = {old: new for new, old in enumerate(nodes)}
        batch_idx = [node_map[n] for n in batch]
        all_preds[batch] = output[batch_idx].argmax(dim=1).cpu()
    
    # FIX: Apply mask to full all_preds
    test_pred = all_preds[test_mask]
    test_true = labels_tensor[test_mask]
    test_acc = accuracy_score(test_true, test_pred)
    test_f1 = f1_score(test_true, test_pred, average='weighted')
    test_f1_macro = f1_score(test_true, test_pred, average='macro')
    
    print(f"\n{'='*80}")
    print("FINAL RESULTS")
    print('='*80)
    print(f"Test Accuracy:      {test_acc:.4f}")
    print(f"Test F1 (weighted): {test_f1:.4f}")
    print(f"Test F1 (macro):    {test_f1_macro:.4f}")
    print('='*80)
    
    print("\nClassification Report:")
    print(classification_report(test_true, test_pred, digits=4))

# Save results
results = {
    'model': 'GCN with Mini-batch Sampling',
    'best_val_accuracy': float(best_val_acc),
    'best_val_f1': float(best_val_f1),
    'test_accuracy': float(test_acc),
    'test_f1_weighted': float(test_f1),
    'test_f1_macro': float(test_f1_macro),
    'num_classes': int(num_classes),
    'n_train': int(train_mask.sum()),
    'n_val': int(val_mask.sum()),
    'n_test': int(test_mask.sum()),
    'hyperparameters': {
        'hidden_dim': config.HIDDEN_DIM,
        'dropout': config.DROPOUT,
        'learning_rate': config.LEARNING_RATE,
        'weight_decay': config.WEIGHT_DECAY,
        'batch_size': config.BATCH_SIZE,
        'num_neighbors': config.NUM_NEIGHBORS,
        'epochs': config.EPOCHS
    }
}

with open(config.RESULTS_PATH, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n✅ Results saved to {config.RESULTS_PATH}")
print(f"✅ Model saved to {config.MODEL_SAVE_PATH}")
print("\n" + "="*80)
print("TRAINING COMPLETE!")
print("="*80)

TRAINING GCN WITH MINI-BATCH SAMPLING

Cleaning GPU memory...
✓ GPU memory cleared
  GPU: NVIDIA GeForce RTX 3050 Laptop GPU
  Memory allocated: 0.0 MB

Device: cuda

LOADING DATA
✓ Nodes: 142,817, Edges: 35,310,216
✓ Features: (142817, 768)
✓ Train: 12,281, Val: 6,141, Test: 104,395
✓ Number of classes: 10
✓ Label range: [0, 9]
✓ Labels already in [0, num_classes-1]
✓ Label validation passed

Preparing tensors...
✓ Tensors prepared

Initializing model...
✓ Model created on CPU
  Input: 768, Hidden: 256, Output: 10
  Parameters: 199,434

Moving model to GPU...
✓ Model on GPU
  GPU memory: 0.8 MB
✓ Optimizer initialized

TRAINING
Training on 12,281 document nodes
Batches per epoch: 24



Epoch   1 | Time: 19.7s | Loss: 1.9872


Epoch   2 | Time: 19.5s | Loss: 1.2549


Epoch   3 | Time: 19.0s | Loss: 1.0507


Epoch   4 | Time: 19.0s | Loss: 0.9604


...

Epoch 198 | Time: 18.3s | Loss: 0.7573


Epoch 199 | Time: 18.8s | Loss: 0.7449


Epoch 200 | Time: 18.7s | Loss: 0.7421 | Val Acc: 0.8870 | Val F1: 0.8701 

✅ Training complete!
Best Val Acc: 0.9228, Best Val F1: 0.9222

TEST EVALUATION


Testing: 100%|███████████████████████████████████████████████████████████████████████| 204/204 [02:11<00:00,  1.55it/s]


FINAL RESULTS
Test Accuracy:      0.9568
Test F1 (weighted): 0.9564
Test F1 (macro):    0.9521

Classification Report:
              precision    recall  f1-score   support

           0     0.9148    0.7493    0.8238      8349
           1     0.9890    0.9861    0.9875     12742
           2     0.9347    0.9961    0.9644     12244
           3     0.9850    0.9678    0.9763      7540
           4     0.8234    0.9465    0.8807     10024
           5     0.9884    0.9671    0.9776     12296
           6     0.9829    0.9771    0.9800     12353
           7     0.9823    0.9838    0.9831     12561
           8     0.9906    0.9263    0.9574      6734
           9     0.9837    0.9972    0.9904      9552

    accuracy                         0.9568    104395
   macro avg     0.9575    0.9497    0.9521    104395
weighted avg     0.9585    0.9568    0.9564    104395


✅ Results saved to gcn_minibatch_results.json
✅ Model saved to best_gcn_minibatch.pt

TRAINING COMPLETE!
